# This file is about cleaning the 'actor' column of traces250102_clean.csv
The output of this file is the input for the anonymizing.py

## Libraries

In [55]:
# ---
# jupyter:
#   jupytext:
#     formats: ipynb,py:light
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.17.0
#   kernelspec:
#     display_name: venv_jupyter
#     language: python
#     name: venv_jupyter
# ---

# # Regarder les identifiants avant anonymisation

# import les bibliothèque
import sys
sys.path.append('../') # ces deux lignes permettent au notebook de trouver le chemin jusqu'au code source contenu dans 'src'
sys.path.append('../src/') # Yvan dit que c'est absurde, mais sans ça ne marche pas
from src import * # on importe ce qui se trouve dans 'src'
import pandas as pd
#from src import script_initialisation, clean_corrupted_data *not used*
from src.data import cleaning
from src.features import utils
import re
import time

# mise à jour Automatiquement
get_ipython().run_line_magic('load_ext', 'autoreload') # pour mise à jour 
get_ipython().run_line_magic('autoreload', '2') # pour mise à jour

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Loading Dataframe

In [56]:
# Read dataframe from load_clean_data() which cleans: timestamp, time_delta, session.duration

df = cleaning.load_clean_data('traces250102_clean.csv')

print(df.head())

print(df.columns)

  research_usage result                  _id.$oid  \
0            1.0         675aa485a7dff6d2d4811b45   
1            1.0         675aa4cea7dff6d2d4811b4f   
2            1.0         675aa4cea7dff6d2d4811b50   
3            1.0         671b54aabd5a98b8f9dc8c14   
4            1.0         671b54f6bd5a98b8f9dc8c45   

                   timestamp.$date              stored.$date            actor  \
0 2024-12-12 09:53:24.624000+00:00  2024-11-27T10:57:29.801Z  abaly.oura.etu/   
1 2024-12-12 09:53:25.008000+00:00  2024-11-27T10:57:29.801Z  abaly.oura.etu/   
2 2024-12-12 09:54:38.793000+00:00  2024-11-27T10:57:29.801Z  abaly.oura.etu/   
3 2024-10-25 10:19:54.344000+00:00  2024-08-29T07:34:11.687Z  abaly.oura.etu/   
4 2024-10-25 10:19:54.583000+00:00  2024-08-29T07:34:11.687Z  abaly.oura.etu/   

  actor.objectType           verb  \
0            Agent  Session.Start   
1            Agent      File.Open   
2            Agent    Session.End   
3            Agent  Session.Start   
4        

## Cleaning process

We want to clean the 'actor' column where present the prenom.nom.etu of students. The problems which we have in this column are:
- Remove '/' from the end of each name
- Remove the binom student, we just want the first student
- Remove '@univ-lille.fr/' part from the name
- Remove all invalid names like nebut or luc

At the end, the column actor should only include the name of student with the 'prenom.nom.etu' format

In [ ]:
# Define the regex pattern

pattern = r'^[a-zA-Z\-]+\.[a-zA-Z0-9\-]+\.etu$'
invalid_names = []

# Apply a cleaning function
def clean_actor(name):
    """
    clean actors with this pattern 'prenom.nom.etu'.

    Args:
        name : A row of DataFrame.

    Returns:
        the cleaned name of actor
    """
     
    #print(name) this is for test
    if isinstance(name, str):
        # remove '/'
        name = name.split('/')

        # remove ' '
        if '' in name : name.remove('')

        # keep only one student
        name = name[0]
    
        #print(name) this is for test
        
        # check pattern prenom.nom.etu
        if re.match(pattern, name):
            return name  # valid, keep as is
        
        elif '@' in name:  # maybe correctable
            match = re.match(r'^([a-zA-Z\-]+\.[a-zA-Z0-9\-]+\.etu)@', name)

            if match:
                return match.group(1)  # extract clean part
        else: 
            invalid_names.append(name)
            
    return None  # discard bad names


## Testing process on a column

In [58]:
# Test function of clean_actor(name)
def test_on_dataframe(df: pd.DataFrame, column: str) -> int:
    """
    Counts the number of values in a DataFrame column that do NOT match
    the pattern 'prenom.nom.etu'.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column to check.

    Returns:
        int: Number of values not matching the expected pattern.
    """

    pattern = r'^[a-zA-Z\-]+\.[a-zA-Z0-9\-]+\.etu$'
    return df[column].apply(lambda x: not re.fullmatch(pattern, str(x).strip())).sum()

## Main function

In [ ]:
# main function to start the process
def main():
    """
    Start the process cleaning and testing.

    Args:
        None

    Returns:
        None
    """
    
    # Before cleaning function
    print("Before applying clean_actor() on df:")
    print(df['actor'])

    # Start test to get the unmatch actors 
    print("\n Test process...")
    # calculate the time spending
    start_time = time.time()
    not_matching = test_on_dataframe(df,'actor')
    end_time = time.time()
    elapsed_time = end_time - start_time 
    print(f"Test done! in {elapsed_time:.4f} seconds")

    print(f"The total number of not matching : {not_matching} \n")

    # Apply cleaning function
    print("Apply cleaning function...")
    # calculate the time spending
    start_time = time.time()
    df['clean_actor'] = df['actor'].apply(clean_actor)
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Apply done! in {elapsed_time:.4f} seconds \n")

    # After cleaning function
    print("After applying clean_actor() on df")
    print(df[['actor','clean_actor']])

    # Start test to get the unmatch actors
    print("Test process...")
    # calculate the time spending
    start_time = time.time()
    not_matching = test_on_dataframe(df,'clean_actor')
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Test done! in {elapsed_time:.4f} seconds")
    print('\n')

    print(f"The total number of not matching : {not_matching} ")

In [60]:
main()

Before applying clean_actor() on df
0                               abaly.oura.etu/
1                               abaly.oura.etu/
2                               abaly.oura.etu/
3                               abaly.oura.etu/
4                               abaly.oura.etu/
                          ...                  
306941    zacharie.deroo.etu/noe.carpentier.etu
306942    zacharie.deroo.etu/noe.carpentier.etu
306943    zacharie.deroo.etu/noe.carpentier.etu
306944    zacharie.deroo.etu/noe.carpentier.etu
306945    zacharie.deroo.etu/noe.carpentier.etu
Name: actor, Length: 306946, dtype: object
Test process...
Test done! in 0.7407 seconds
The total number of not matching : 306946 

Apply cleaning function...
Apply done!in {elapsed_time:.4f} seconds
After applying clean_actor() on df
                                        actor         clean_actor
0                             abaly.oura.etu/      abaly.oura.etu
1                             abaly.oura.etu/      abaly.oura.etu
2  